## Typicality rating experiment — LLaMA 3.1 8B Instruct

In [18]:
import os
# Must be set BEFORE importing transformers, or it will still probe TensorFlow
#os.environ["USE_TF"] = "0"
#os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import re
import pandas as pd
import torch
from transformers import pipeline

#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/"  # change to "meta-llama/Meta-Llama-3-8B-Instruct" if you need 3.0

In [19]:
# Set the language to run this notebook for.
# Must match the suffix used in your CSV's instance_<LANGUAGE> column,
# e.g. "English", "German", "Spanish"
LANGUAGE = "german"

instance_col = LANGUAGE
norm_rating_col = f"norm_rating_{LANGUAGE}"


In [20]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

5

In [21]:
df = pd.read_csv('typicalities_dataset.csv')
df.head(5)
df = df[df[instance_col].notna() & (df[instance_col].astype(str).str.strip() != "")].reset_index(drop=True)

2355

824

499

In [22]:
df

,Unnamed: 0,concept,spanish,spanish_to_eng,german,german_to_eng,english,typ_es,typ_de,typ_en
0,3,animal,NaN,NaN,Ameise,ant,ant,NaN,0.708333,0.340292
1,5,animal,NaN,NaN,Antilope,antelope,antelope,NaN,0.675000,0.552682
2,6,animal,NaN,NaN,Dachs,badger,badger,NaN,0.691667,0.544778
3,8,animal,NaN,NaN,Fledermaus,bat,bat,NaN,0.708333,0.520181
4,9,animal,Oso,bear,Bär,bear,bear,0.946667,0.966667,0.838305
...,...,...,...,...,...,...,...,...,...,...
819,3177,vehicle,NaN,NaN,Kahn,rowing boat,NaN,NaN,0.433333,NaN
820,3178,vehicle,NaN,NaN,Segelschiff,sailing ship,NaN,NaN,0.558333,NaN
821,3179,vehicle,NaN,NaN,Schlitten,sleigh,NaN,NaN,0.366667,NaN
822,3180,vehicle,NaN,NaN,Straßenbahn,tram,NaN,NaN,0.941667,NaN


In [24]:
def typicality_prompt(obj, category):
    return f"""
Eres un experto en psicología cognitiva.
Evalúe hasta qué punto el elemento "{obj}" es típico de la categoría "{category}".
Responda ÚNICAMENTE con un número entero entre 1 y 7, donde 1 significa "nada típico" y 7 significa "extremadamente típico".
"""

In [12]:
def typicality_prompt(obj, category):
    return f"""
You are a cognitive psychology expert. 
Rate how typical the item "{obj}" is of the category "{category}".
Respond with ONLY one integer from 1 to 7, where 1 means "not typical at all" and 7 means "extremely typical"."""

In [23]:
def typicality_prompt(obj, category):
    return f"""
Du bist Experte für kognitive Psychologie.
Bewerte, wie typisch das Element "{obj}" für die Kategorie "{category}" ist.
Antworte NUR mit einer einzigen ganzen Zahl von 1 bis 7, wobei 1 "überhaupt nicht typisch" und 7 "äußerst typisch" bedeutet.
"""

In [24]:
def get_typicality(obj, category):
    prompt = typicality_prompt(obj, category)
    messages = [
        {"role": "user", "content": prompt},
    ]
    output = pipe(
        messages,
        max_new_tokens=10,
        do_sample=False,
    )
    response = output[0]["generated_text"][-1]["content"]
    match = re.search(r"[1-7]", response)
    if match:
        return int(match.group())
    else:
        print(f"Could not parse response for '{obj}' ({category}): {response!r}")
        return None


In [9]:
ls /data1/shared_models/models--google--gemma-3-12b-it/snapshots/

96b6f1eccf38110c56df3a15bffe176da04bfd80/


In [25]:
from transformers import pipeline
#MODEL_NAME = "/data1/shared_models/models--google--gemma-3-12b-it/snapshots/96b6f1eccf38110c56df3a15bffe176da04bfd80//"
MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


In [26]:
# Reset column in case a previous buggy run stored prompt text instead of scores
df["llama_typicality"] = pd.to_numeric(df["llama_typicality"], errors="coerce") if "llama_typicality" in df.columns else None

output_path = f"combined_prototypes_llama_likert_{LANGUAGE}.csv"

for idx, row in df.iterrows():
    if pd.notna(df.at[idx, "llama_typicality"]):
        continue  # already done, skip (useful on resume)
    score = get_typicality(row[instance_col], row["concept"])
    df.at[idx, "llama_typicality"] = score
    df.to_csv(output_path, index=False)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [27]:
df.to_csv(f"llama_{LANGUAGE.lower()}_likert.csv", index=False)